### Install if needed the below

In [11]:
# If needed please install required packages below
# pip install -U ultralytics opencv-python pyyaml matplotlib

### Imports + Paths

In [12]:
# Script to create a YOLOv8 dataset from augmented chatbot screenshots
# and additional negative samples for robustness testing.
import os
import csv
import glob
import random
import shutil
import yaml
from pathlib import Path

# Libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

random.seed(42) 

# Input from Task1
AUG_DIR = Path("output_aug") # Augmented chatbot screenshots
CSV_PATH = AUG_DIR / "labels.csv" # CSV with bounding box annotations

# Additional negative samples
NEG_DIR = Path("negatives") # Directory with negative samples

# YOLO dataset output
YOLO_DIR = Path("yolo_dataset") # Output YOLOv8 dataset directory
IMG_DIR = YOLO_DIR / "images" # Images directory
LBL_DIR = YOLO_DIR / "labels" # Labels directory

CLASSES = ["chatgpt", "gemini", "claude"] # Classes of chatbot screenshots
CLASS_TO_ID = {c:i for i,c in enumerate(CLASSES)} # Class to ID mapping

SPLIT = {"train":0.70, "val":0.20, "test":0.10} # Dataset split ratios

print("AUG_DIR:", AUG_DIR.resolve()) # Print absolute path
print("CSV_PATH exists:", CSV_PATH.exists()) # Check if CSV exists
print("NEG_DIR exists:", NEG_DIR.exists()) # Check if negatives directory exists

AUG_DIR: C:\IPCV\Homeassignment\ui-detector\output_aug
CSV_PATH exists: True
NEG_DIR exists: True


### Sanity checks

In [13]:
assert CSV_PATH.exists(), "Missing output_aug/labels.csv. Run Task1 notebook first." # Check CSV
assert len(list(AUG_DIR.glob("*.jpg"))) > 0, "No images in output_aug/. Run Task1 notebook first." # Check images

### YOLO Label Conversion

This function converts bounding box coordinates from pixel format to YOLO format:

1. Pixel to Normalized: Converts from (x1,y1,x2,y2) pixel coordinates to YOLO's (center_x, center_y, width, height) normalized format
2. Normalization: Divides all values by image dimensions (W,H) to get values between 0-1
3. Bounds Clamping: Ensures coordinates stay within [0,1] range to prevent training issues
4. File Output: Writes single-line label file with class ID and normalized coordinates (6 decimal precision)

**Purpose**: Converts bounding box annotations to the format required by YOLO object detection models, enabling direct use of generated images for training.

In [14]:
# Function to write YOLO label file
def write_yolo_label(label_path: Path, class_id: int, x1,y1,x2,y2, W,H):
    # convert xyxy pixel -> yolo normalized cx cy w h
    bw = max(1, x2 - x1)    
    bh = max(1, y2 - y1)    
    cx = x1 + bw/2.0    
    cy = y1 + bh/2.0    

    cx /= W 
    cy /= H 
    bw /= W 
    bh /= H 

    # clamp
    cx = min(max(cx, 0.0), 1.0)  
    cy = min(max(cy, 0.0), 1.0)  
    bw = min(max(bw, 0.0), 1.0) 
    bh = min(max(bh, 0.0), 1.0) 

    label_path.write_text(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n", encoding="utf-8") 

### Read CSV

In [15]:
# Read CSV annotations
rows = []   
with open(CSV_PATH, "r", encoding="utf-8") as f:
    r = csv.DictReader(f)   
    for row in r:           
        rows.append(row)    

# Print number of rows read
print("CSV rows:", len(rows))   
assert len(rows) > 0            

CSV rows: 360


### Dataset Preparation for YOLO

This code sets up the directory structure and splits the data for YOLO model training:

1. Cleanup: Removes existing YOLO dataset directory to start fresh
2. Directory Creation: Creates main image/label directories and subdirectories for train/val/test splits
3. Data Shuffling: Randomizes the dataset to ensure unbiased splits
4. Data Splitting: Divides data according to predefined ratios (typically 70% train, 20% validation, 10% test)
5. Size Verification: Prints counts for each split to confirm distribution

**Purpose**: Prepares organized, properly split datasets required by YOLO training pipelines, ensuring clean separation of training, validation, and test data.

In [16]:
# Clean old dataset
if YOLO_DIR.exists():                           
    shutil.rmtree(YOLO_DIR)                     
IMG_DIR.mkdir(parents=True, exist_ok=True)   
LBL_DIR.mkdir(parents=True, exist_ok=True)   

for split in ["train","val","test"]:        
    (IMG_DIR/split).mkdir(parents=True, exist_ok=True)  
    (LBL_DIR/split).mkdir(parents=True, exist_ok=True)  

# Shuffle and split data
random.shuffle(rows)    
n = len(rows)   
n_train = int(n * SPLIT["train"])   
n_val   = int(n * SPLIT["val"])    

# Split rows into train, val, test sets
train_rows = rows[:n_train]  
val_rows   = rows[n_train:n_train+n_val]    
test_rows  = rows[n_train+n_val:]       

# Print split sizes for verification
print("split sizes:", len(train_rows), len(val_rows), len(test_rows)) 

split sizes: 251 72 37


### Dataset Processing for YOLO Training

This function processes each dataset split (train/val/test) to create YOLO compatible files:

1. Image Copying: Copies augmented images to appropriate split directories
2. Label Generation: Creates YOLO format label files with normalized bounding box coordinates
3. Missing Image Handling: Counts and reports images referenced in CSV but missing from disk
4. Dimension Extraction: Reads each image to get height/width for proper coordinate normalization
5. Class ID Mapping: Converts class names to numeric IDs using CLASS_TO_ID dictionary

**Purpose**: Transforms the generated augmented data into the exact file structure and format required by YOLO training scripts, ensuring all images and labels are properly organized and formatted.

In [17]:
# Process each split and copy images/labels to respective directories
def process_split(split_name, split_rows):
    missing = 0                   
    for row in split_rows:      
        fname = row["filename"]  
        cls   = row["class"]

        # Copy image and write label file
        src_img = AUG_DIR / fname       
        if not src_img.exists():        
            missing += 1                
            continue

        # Destination paths for image and label files
        dst_img = IMG_DIR / split_name / fname  
        dst_lbl = LBL_DIR / split_name / (Path(fname).stem + ".txt")    

        # Copy image to destination directory
        shutil.copy2(src_img, dst_img)  

        # Read image to get dimensions 
        img = cv2.imread(str(dst_img))    
        H, W = img.shape[:2]        

        # Get bounding box coordinates from CSV row 
        x1 = int(row["x1"]); y1 = int(row["y1"]); x2 = int(row["x2"]); y2 = int(row["y2"])  
        class_id = CLASS_TO_ID[cls]                                                         
        write_yolo_label(dst_lbl, class_id, x1,y1,x2,y2, W,H)                   

    # Print number of missing images for this split 
    print(f"{split_name}: missing images referenced in CSV = {missing}")    

# Process each split and create dataset directories
process_split("train", train_rows)  
process_split("val",   val_rows)    
process_split("test",  test_rows)   

train: missing images referenced in CSV = 0
val: missing images referenced in CSV = 0
test: missing images referenced in CSV = 0


### Adding Negative Samples

This code adds images without target objects (negative samples) to improve model robustness:

1. Image Collection: Gathers negative images from a specified directory using the list_images helper
2. Quantity Control: Limits negative samples to prevent dataset imbalance (default 250, called with 300)
3. Proportional Splitting: Distributes negatives across train/val/test using the same ratios as positive data
4. Empty Label Files: Creates blank label files for negatives (no bounding boxes to detect)
5. Directory Integration: Copies negative images into the same split directories as positive examples

**Purpose**: Prevents the model from becoming overly sensitive by providing examples with no objects to detect, which helps reduce false positives and improves generalization to real world scenarios.

In [18]:
# Function to list image files in a folder 
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}   

# List images in a folder
def list_images(folder: Path):  
    if not folder.exists():     
        return []               
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS])   

# Add negative samples to dataset
def add_negatives(max_neg=250):  
    negs = list_images(NEG_DIR)  # List negative images
    print("Found negatives:", len(negs))    
    if len(negs) == 0:          
        print("No negatives found. Skipping.")  
        return                  
    
    # Shuffle and limit negatives to max_neg
    random.shuffle(negs)    
    negs = negs[:max_neg]   

    # Split negatives into train, val, test sets
    n = len(negs)   
    n_train = int(n * SPLIT["train"])   
    n_val   = int(n * SPLIT["val"])   
    train_set = negs[:n_train]      
    val_set   = negs[n_train:n_train+n_val]  
    test_set  = negs[n_train+n_val:]    

    # Copy negatives and create empty label files
    for split_name, files in [("train",train_set), ("val",val_set), ("test",test_set)]: 
        for src in files:       
            dst_img = IMG_DIR / split_name / src.name   
            dst_lbl = LBL_DIR / split_name / (src.stem + ".txt")    

            # Copy negative image and create empty label file
            shutil.copy2(src, dst_img)  
            dst_lbl.write_text("", encoding="utf-8")  

    # Print number of negatives added to each split 
    print("Added negatives:", len(train_set), len(val_set), len(test_set))  

# Add negatives to dataset
add_negatives(max_neg=300) 

Found negatives: 3
Added negatives: 2 0 1


### YOLO Configuration File Creation

This code generates the data.yaml configuration file required by YOLO training:

1. Path Structure: Defines base dataset directory and subdirectories for train/val/test splits
2. Class Mapping: Creates a dictionary mapping numeric IDs to class names (e.g., {0: "chatgpt", 1: "copilot"})
3. YAML Formatting: Writes the configuration in YAML format using yaml.safe_dump()
4. Verification: Prints the generated file contents for immediate review

**Purpose**: Creates the essential configuration file that tells YOLO where to find training data, how many classes exist, and what each class is called - a critical step before starting model training.

In [19]:
# Write data.yaml file for YOLOv8 training
data = {
    "path": str(YOLO_DIR),  # Base path for dataset
    "train": "images/train",    # Training images subdir
    "val": "images/val",    # Validation images subdir
    "test": "images/test",   # Test images subdir
    "names": {i: c for i,c in enumerate(CLASSES)}   # Class names mapping
}

# Write data.yaml file 
with open("data.yaml", "w", encoding="utf-8") as f:  # Open data.yaml for writing
    yaml.safe_dump(data, f, sort_keys=False)        # Write YAML data to file


print("Wrote data.yaml")    # Print confirmation
print(open("data.yaml", "r", encoding="utf-8").read())  # Print contents of data.yaml

Wrote data.yaml
path: yolo_dataset
train: images/train
val: images/val
test: images/test
names:
  0: chatgpt
  1: gemini
  2: claude



### Training configuration for my model

In [20]:
# Training configuration 
PROJECT_DIR = "runs_chatbot" # Directory to save training runs
RUN_NAME = "task4_v1"    # Name of this training run

EPOCHS = 50    # Epochos are there for training stability
IMGSZ  = 960   # Image size for training
BATCH  = 16    # Batch size

DEVICE = 0      # 0 for GPU, "cpu" for CPU
WORKERS = 0     # Workers are the number of subprocesses for data loading
CACHE = False   # cache dataset in RAM for faster training. FALSE for now. Set TRUE after stable
PATIENCE = 15   # Early stopping patience. Patience is there to prevent overfitting.

### Train yolov8n

In [21]:
model = YOLO("yolov8n.pt")  # start with nano, switch to yolov8s.pt for better box quality if needed

results = model.train( 
    data="data.yaml",       
    epochs=EPOCHS,          
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache=CACHE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=PATIENCE
)


New https://pypi.org/project/ultralytics/8.4.7 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.248  Python-3.13.7 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=task4_v1, nbs=64, nms=False, opset=Non

### Loading Trained Model Weights

This code locates and loads the best performing model weights from YOLO training runs:

1. Weight Discovery: Searches for the best weights (best.pt) in the expected YOLO output directory structure
2. Fallback Logic: If best weights aren't found, tries last.pt or searches recursively for the most recent best.pt
3. Model Initialization: Loads the trained YOLO model with the discovered weights
4. Validation: Asserts that weights exist before loading to prevent runtime errors

**Purpose**: Retrieves the optimal model checkpoint from previous training sessions for inference or further training, ensuring the highest accuracy model is used.

In [22]:
# Load best model weights after training 
def find_weights(project_dir, run_name):    # Find best or last weights file 
    expected_best = os.path.join(project_dir, run_name, "weights", "best.pt")   
    expected_last = os.path.join(project_dir, run_name, "weights", "last.pt")   

    # Check expected paths
    if os.path.exists(expected_best):
        return expected_best    
    if os.path.exists(expected_last):
        return expected_last    

    # fallback: latest best.pt under project_dir
    candidates = glob.glob(os.path.join(project_dir, "**", "weights", "best.pt"), recursive=True)   
    if candidates:  
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)    
        return candidates[0]    

    return None 

# Load best weights
weights_path = find_weights(PROJECT_DIR, RUN_NAME)  
assert weights_path is not None, "Could not find weights. Check runs_chatbot folder." 

# Load model with best weights
print("Loading:", weights_path) 
best = YOLO(weights_path)  

Loading: runs_chatbot\task4_v1\weights\best.pt


### Model Evaluation on Test Set

This code evaluates the trained model's performance using the test dataset:

1. Test Set Evaluation: Uses the best.val() method to run inference on the test split
2. Configuration: Points to the same data.yaml file used for training to maintain consistency
3. Device Specification: Runs evaluation on the specified device (GPU/CPU) for optimal performance
4. Metrics Extraction: Returns comprehensive performance metrics including precision, recall, mAP, etc.

**Purpose**: Measures the model's real-world performance on unseen data to validate training effectiveness, detect overfitting, and establish baseline performance metrics.

In [23]:
# Evaluate on test set
metrics = best.val(data="data.yaml", split="test", device=DEVICE)
metrics

Ultralytics 8.3.248  Python-3.13.7 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1249.4433.7 MB/s, size: 169.8 KB)
val: Scanning C:\IPCV\Homeassignment\ui-detector\yolo_dataset\labels\test... 38 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 692.4it/s 0.1s
val: New cache created: C:\IPCV\Homeassignment\ui-detector\yolo_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0s/it 3.1s1.0ss
                   all         38         37      0.981      0.988      0.995      0.981
               chatgpt         18         18      0.995          1      0.995      0.989
                gemini          7          7      0.948          1      0.995      0.995
                claude         12         12          1      0.964      0.995      0.959
Spe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000234AD456040>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

### Sample Predictions on Test Images

This code generates and saves predictions for a random subset of test images:

1. Image Selection: Randomly picks 6 images from the test set directory
2. Model Inference: Runs the trained YOLO model on these images with a 0.50 confidence threshold
3. Result Saving: Saves the prediction images to a designated output directory (pred_samples/test_samples/)
4. Directory Management: Uses exist_ok=True to avoid errors if the directory already exists

**Purpose**: Provides visual examples of model performance on test data, allowing for qualitative assessment of detection accuracy, bounding box quality, and false positive/negative rates.

In [24]:
test_imgs = list((IMG_DIR/"test").glob("*.jpg"))
random.shuffle(test_imgs)
sample = [str(p) for p in test_imgs[:6]]

best.predict(source=sample, conf=0.50, save=True, project="pred_samples", name="test_samples", exist_ok=True)
print("Saved sample preds to pred_samples/test_samples/")



0: 960x960 1 claude, 6.5ms
1: 960x960 1 chatgpt, 6.5ms
2: 960x960 1 gemini, 6.5ms
3: 960x960 1 gemini, 6.5ms
4: 960x960 1 chatgpt, 6.5ms
5: 960x960 1 chatgpt, 1 gemini, 6.5ms
Speed: 4.0ms preprocess, 6.5ms inference, 1.1ms postprocess per image at shape (1, 3, 960, 960)
Results saved to C:\IPCV\Homeassignment\ui-detector\pred_samples\test_samples
Saved sample preds to pred_samples/test_samples/


### Video Inference for Real-World Testing

This code runs the trained model on a video file to test performance in dynamic scenarios:

1. Video File Detection: Tries multiple video formats (.avi then .mp4) for compatibility
2. Parameter Configuration: Uses specific settings optimized for video (0.55 confidence, 0.35 IoU)
3. Performance Optimization: Sets image size (IMGSZ) for consistent processing and limits to 8 detections per frame
4. Output Management: Saves annotated video to video_preds/exam_clip/ directory

**Purpose**: Tests model performance on real-world video data, simulating how it would perform in production scenarios with moving content and varying lighting conditions.

In [21]:
video_path = "exam_clip.avi"
if not os.path.exists(video_path):
    video_path = "exam_clip.mp4"

assert os.path.exists(video_path), "Put exam_clip.avi/mp4 next to this notebook."

best.predict(
    source=video_path,
    conf=0.55,
    iou=0.35,
    imgsz=IMGSZ,
    max_det=8,
    save=True,
    project="video_preds",
    name="exam_clip",
    exist_ok=True
)

print("Saved to video_preds/exam_clip/")


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



Results saved to C:\IPCV\Homeassignment\ui-detector\video_preds\exam_clip
Saved to video_preds/exam_clip/


### Video Cleaning with Intelligent Filtering (Cleaner video output)

This code creates a cleaner video output by applying advanced filters to model predictions:

1. Video Processing Pipeline: Reads input video frame by frame and applies detection with optimized parameters
2. Geometric Filtering: Removes unrealistic detections based on:
   - Size Ratio: Filters boxes that are too small (<2% of frame) or too large (>75% of frame)
   - Aspect Ratio: Excludes boxes with extreme width:height ratios (>5:1 or 1:5)
3. Best Per Class Selection: Keeps only the highest confidence detection for each class per frame
4. Quality Output: Creates a clean MP4 video with consistent formatting and minimal false positives
5. Visual Enhancement: Uses yellow bounding boxes with class labels for clear visibility

**Purpose**: Produces professional, production-ready video output by filtering out noisy detections and focusing on the most confident, geometrically plausible predictions for each frame.

In [ ]:
import os
import cv2
import numpy as np

# Clean video predictions by keeping only best box per class per frame
out_dir = "video_preds_clean"
os.makedirs(out_dir, exist_ok=True)
out_video = os.path.join(out_dir, "exam_clip_clean.mp4")

# Open input video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Prepare video writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(out_video, fourcc, fps, (W, H))

# Detection parameters
CONF = 0.55
IOU  = 0.35

# Geometry filters
MIN_AREA_RATIO = 0.02
MAX_AREA_RATIO = 0.75
MAX_AR = 5.0

# Class names
names = best.names

# Geometry functions
def area_ratio(x1,y1,x2,y2):
    return ((x2-x1)*(y2-y1))/float(W*H)

# Aspect ratio function
def max_aspect_ratio(x1,y1,x2,y2):
    w = max(1, x2-x1)
    h = max(1, y2-y1)
    return max(w/h, h/w)

#   Process video frames
while True:
    ret, frame = cap.read()
    if not ret:
        break

    r = best.predict(frame, conf=CONF, iou=IOU, imgsz=IMGSZ, max_det=20, verbose=False)[0]
    best_per_class = {}

    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.cpu().numpy().astype(int)
        cls  = r.boxes.cls.cpu().numpy().astype(int)
        conf = r.boxes.conf.cpu().numpy()

        for (x1,y1,x2,y2), c, s in zip(xyxy, cls, conf):
            x1 = max(0, min(W-1, x1)); x2 = max(0, min(W, x2))
            y1 = max(0, min(H-1, y1)); y2 = max(0, min(H, y2))
            if x2 <= x1+2 or y2 <= y1+2:
                continue

            a = area_ratio(x1,y1,x2,y2)
            ar = max_aspect_ratio(x1,y1,x2,y2)

            if a < MIN_AREA_RATIO or a > MAX_AREA_RATIO:
                continue
            if ar > MAX_AR:
                continue

            if (c not in best_per_class) or (s > best_per_class[c][0]):
                best_per_class[c] = (s, (x1,y1,x2,y2))

    for c, (s, (x1,y1,x2,y2)) in best_per_class.items():
        label = f"{names[c]} {s:.2f}"
        cv2.rectangle(frame, (x1,y1), (x2,y2), (255,255,0), 3)
        cv2.putText(frame, label, (x1, max(25, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,0), 2)

    writer.write(frame)

cap.release()
writer.release()

print("Saved clean video to:", out_video)

Saved clean video to: video_preds_clean\exam_clip_clean.mp4
